In [1]:
from Bio.PDB import PDBParser
import os

In [ ]:
files = os.listdir('./PepSet/Merged_PDBs')
for file in files:
    if file.endswith('.pdb'):
        file_path = os.path.join('./PepSet/Merged_PDBs', file)
        structure = PDBParser(QUIET=True).get_structure(file[:-4], file_path)
        for model in structure:
            chain_count = len(model)
            if chain_count != 2:
                print(f"{file} has {chain_count} chains.")

3hds.pdb has 3 chains.
2fym.pdb has 3 chains.
1d8d.pdb has 3 chains.
2ds8.pdb has 3 chains.
3aa0.pdb has 3 chains.
6cwp.pdb has 3 chains.
4oz1.pdb has 3 chains.
4qh7.pdb has 3 chains.
3av9.pdb has 3 chains.
1vpp.pdb has 3 chains.
5ow5.pdb has 3 chains.


In [5]:
# 检查/home/junjiechen/1_work/250401-Dpepalign/Benchmark/ligandmpnn/test_weight/datasets/PepSet/Merged_PDBs目录下所有的pdb文件与/home/junjiechen/1_work/250401-Dpepalign/Benchmark/ligandmpnn/test_weight/datasets/PepSet/PepSet-bound目录下所有pdb的文件内残基每个原子的坐标是否一致
import os
from Bio.PDB import PDBParser
def compare_pdb_files(file1, file2):
    parser = PDBParser(QUIET=True)
    structure1 = parser.get_structure('structure1', file1)
    structure2 = parser.get_structure('structure2', file2)

    # 只比较第一条链的骨架原子
    # 仅比较第一模型中的第一条链的骨架原子
    model1 = next(structure1.get_models(), None)
    model2 = next(structure2.get_models(), None)
    chain1 = next(model1.get_chains(), None) if model1 is not None else None
    chain2 = next(model2.get_chains(), None) if model2 is not None else None

    if chain1 is None or chain2 is None:
        print("One of the structures has no chain in the first model.")
        return False

    # 按“残基编号+残基名”建立映射，只比较标准残基（排除水/配体等）
    residues1 = {(res.get_id(), res.get_resname()): res for res in chain1 if res.id[0] == ' '}
    residues2 = {(res.get_id(), res.get_resname()): res for res in chain2 if res.id[0] == ' '}

    common_keys = residues1.keys() & residues2.keys()
    atoms1, atoms2 = [], []

    # 比较相同残基中：结构1里在结构2中也存在的原子
    for key in common_keys:
        res1 = residues1[key]
        res2 = residues2[key]
        atom_map2 = {atom.get_name(): atom for atom in res2.get_atoms()}

        for atom1 in res1.get_atoms():
            atom2 = atom_map2.get(atom1.get_name())
            if atom2 is not None:
                atoms1.append(atom1)
                atoms2.append(atom2)

    if not atoms1:
        print("No common residue/atom pairs found between structure1 and structure2.")
        return False

    if len(atoms1) != len(atoms2):
        print(f"Number of atoms differ: {len(atoms1)} vs {len(atoms2)}")
        return False

    for atom1, atom2 in zip(atoms1, atoms2):
        if atom1.get_coord().tolist() != atom2.get_coord().tolist():
            print(f"Coordinates differ for atom {atom1.get_full_id()} and {atom2.get_full_id()}")
            return False

    return True
merged_pdb_dir = '/home/junjiechen/1_work/250401-Dpepalign/Benchmark/ligandmpnn/test_weight/datasets/PepSet/test'
bound_pdb_dir = '/home/junjiechen/1_work/250401-Dpepalign/Benchmark/ligandmpnn/test_weight/datasets/PepSet/PepSet-bound'
for file in os.listdir(merged_pdb_dir):
    if file.endswith('.pdb'):
        file_name = file[:-4]
        merged_pdb_path = os.path.join(merged_pdb_dir, file)
        bound_pdb_path = os.path.join(bound_pdb_dir, file_name + f'/rec_b.pdb')
        if not os.path.exists(bound_pdb_path):
            print(f"File {file} does not exist in {bound_pdb_dir}")
            continue
        if not compare_pdb_files(merged_pdb_path, bound_pdb_path):
            print(f"Files {file} differ between {merged_pdb_dir} and {bound_pdb_dir}")


Coordinates differ for atom ('structure1', 0, 'A', (' ', 16, ' '), ('OG1', ' ')) and ('structure2', 0, 'A', (' ', 16, ' '), ('OG1', ' '))
Files 2fka.pdb differ between /home/junjiechen/1_work/250401-Dpepalign/Benchmark/ligandmpnn/test_weight/datasets/PepSet/test and /home/junjiechen/1_work/250401-Dpepalign/Benchmark/ligandmpnn/test_weight/datasets/PepSet/PepSet-bound
File pep.pdb does not exist in /home/junjiechen/1_work/250401-Dpepalign/Benchmark/ligandmpnn/test_weight/datasets/PepSet/PepSet-bound
File rec_b.pdb does not exist in /home/junjiechen/1_work/250401-Dpepalign/Benchmark/ligandmpnn/test_weight/datasets/PepSet/PepSet-bound


In [ ]:
for root, dir, file in os.walk(".")